In [20]:
import importlib
import numpy as np
import torch
from torch.nn.utils import parameters_to_vector

from common import local_rlct_estimater
import objective_function.linear_dnn as linear_dnn

importlib.reload(local_rlct_estimater)
importlib.reload(linear_dnn)

LocalRLCTTorchEstimator = local_rlct_estimater.LocalRLCTTorchEstimator
make_true_model = linear_dnn.make_true_model
make_learning_model = linear_dnn.make_learning_model
sample_from_true_model = linear_dnn.sample_from_true_model


In [21]:
torch.manual_seed(0)
np.random.seed(0)

# Change these settings from the notebook to control the experiment.
input_dim = 10
true_hidden_dims = (10, 5, 10, 10)
learning_hidden_dims = true_hidden_dims
output_dim = 10
n_samples =500
noise_std = 0.05

print(f"true architecture: {input_dim} -> {true_hidden_dims} -> {output_dim}")
print(f"learning architecture: {input_dim} -> {learning_hidden_dims} -> {output_dim}")


true architecture: 10 -> (10, 5, 10, 10) -> 10
learning architecture: 10 -> (10, 5, 10, 10) -> 10


In [22]:
true_model_for_count = make_true_model(
    input_dim=input_dim,
    hidden_dims=true_hidden_dims,
    output_dim=output_dim,
    dtype=torch.float64,
)
learning_model_for_count = make_learning_model(
    input_dim=input_dim,
    hidden_dims=learning_hidden_dims,
    output_dim=output_dim,
    dtype=torch.float64,
)

true_num_params = sum(param.numel() for param in true_model_for_count.parameters())
learning_num_params = sum(param.numel() for param in learning_model_for_count.parameters())

print(f"true model parameter count: {true_num_params}")
print(f"learning model parameter count: {learning_num_params}")


true model parameter count: 445
learning model parameter count: 445


In [23]:
x = torch.empty(n_samples, input_dim, dtype=torch.float64).uniform_(-2.0, 2.0)

true_model = make_true_model(
    input_dim=input_dim,
    hidden_dims=true_hidden_dims,
    output_dim=output_dim,
    dtype=torch.float64,
)
true_model.eval()
for parameter in true_model.parameters():
    parameter.requires_grad_(False)

y_true, y = sample_from_true_model(true_model, x, noise_std=noise_std)

print(f"x shape: {tuple(x.shape)}")
print(f"y_true shape: {tuple(y_true.shape)}")
print(f"y shape: {tuple(y.shape)}")
print("first 5 samples:")
for i in range(5):
    print(f"x={x[i].tolist()}")
    print(f"y_true={y_true[i].tolist()}")
    print(f"y={y[i].tolist()}")


x shape: (500, 10)
y_true shape: (500, 10)
y shape: (500, 10)
first 5 samples:
x=[1.3054886727037203, -1.3255818838180433, -0.9596986829476397, 1.7944682544626418, -1.3814905451277202, -0.9130049653456558, -1.268304953259404, 0.4031100524234872, -1.502211930301533, -0.46114942814732673]
y_true=[-0.23824493347756653, -0.3887600858270337, -0.11123281652664105, 0.0969029290557247, -0.1545734458939551, -0.29530063596221057, 0.06635394042100831, -0.1757016252687733, 0.3238869632737267, -0.3909745310480792]
y=[-0.28455946034693613, -0.4616740514094927, -0.10789915699609895, 0.07040877969641848, -0.13237260679009305, -0.29579546232721426, 0.13989103185912422, -0.22149488426334368, 0.33987150136829647, -0.42149878609632413]
x=[0.38991679625375886, -1.611607621490439, -1.7623247821435282, 0.46704783541466854, -1.5880504705410035, -1.2674966025702146, -1.9755135785524756, -1.9911333861010196, -1.3519979000204199, -0.5597168646079824]
y_true=[-0.21092815284569913, -0.39634332812287165, -0.1105472

In [24]:
torch.manual_seed(1)

learning_model = make_learning_model(
    input_dim=input_dim,
    hidden_dims=learning_hidden_dims,
    output_dim=output_dim,
    dtype=torch.float64,
)

optimizer = torch.optim.Adam(learning_model.parameters(), lr=5e-3)
loss_history = []

for epoch in range(2500):
    optimizer.zero_grad()
    prediction = learning_model(x)
    loss = torch.mean((prediction - y) ** 2)
    loss.backward()
    optimizer.step()
    loss_history.append(loss.item())

with torch.no_grad():
    fitted = learning_model(x)
    train_mse = torch.mean((fitted - y) ** 2).item()
    true_mse = torch.mean((fitted - y_true) ** 2).item()

w0 = parameters_to_vector([param.detach().clone() for param in learning_model.parameters()])

print(f"final_train_mse={train_mse:.8f}")
print(f"mse_to_true_function={true_mse:.8f}")
print(f"num_parameters={w0.numel()}")
print(f"last_loss={loss_history[-1]:.8f}")
print("first 5 fitted values:")
for i in range(5):
    print(f"pred={fitted[i].tolist()}")
    print(f"y={y[i].tolist()}")
    print(f"y_true={y_true[i].tolist()}")


final_train_mse=0.00253139
mse_to_true_function=0.00003795
num_parameters=445
last_loss=0.00253142
first 5 fitted values:
pred=[-0.23271266907221533, -0.38867161629599206, -0.11344179206394348, 0.11271956684749482, -0.16182362567994812, -0.2930685607560335, 0.07038299123681631, -0.17585139362281468, 0.32267166635252786, -0.3864504774283489]
y=[-0.28455946034693613, -0.4616740514094927, -0.10789915699609895, 0.07040877969641848, -0.13237260679009305, -0.29579546232721426, 0.13989103185912422, -0.22149488426334368, 0.33987150136829647, -0.42149878609632413]
y_true=[-0.23824493347756653, -0.3887600858270337, -0.11123281652664105, 0.0969029290557247, -0.1545734458939551, -0.29530063596221057, 0.06635394042100831, -0.1757016252687733, 0.3238869632737267, -0.3909745310480792]
pred=[-0.20537863528437808, -0.4029688930382039, -0.11099668524135967, 0.0800011173185525, -0.15881582026860247, -0.28161521739314205, 0.06663047782595398, -0.1721497431790137, 0.3062802297104677, -0.35992486931465417]


In [25]:
def mse_loss(model, xb, yb):
    return torch.mean((model(xb) - yb) ** 2)


estimator = LocalRLCTTorchEstimator.from_tensors(
    learning_model,
    mse_loss,
    x,
    y,
    w0=w0,
    eval_mode=True,
)

result = estimator.estimate(
    betas=[4, 8, 16, 32, 64],
    step_size=5e-5,
    n_steps=1500,
    burn_in=400,
    thinning=10,
    clip_radius=3.0,
    grad_clip=75.0,
    n_chains=3,
    max_beta_step=0.05,
    regression_tail=None,
    use_weighted_regression=True,
    update_batch_size=128,
    seed=0,
)

print(f"estimated_lambda={result.lambda_hat:.6f}")
print(f"betaEf={result.betaEf}")
print(f"betaEf_se={result.betaEf_se}")
result.as_dict()


estimated_lambda=34.522649
betaEf=[16.24298251 20.18121988 23.55866232 29.66007041 44.01905546]
betaEf_se=[0.18597232 0.20311903 0.2454198  0.30063307 1.25262136]


{'lambda_hat': 34.5226490053225,
 'betas': array([ 4.,  8., 16., 32., 64.]),
 'betaEf': array([16.24298251, 20.18121988, 23.55866232, 29.66007041, 44.01905546]),
 'mean_f': array([4.06074563, 2.52265248, 1.4724164 , 0.9268772 , 0.68779774]),
 'ess_like_counts': array([330, 330, 330, 330, 330]),
 'x0': array([-2.33004218e-01, -3.13145770e-01, -3.74975381e-02, -3.32546889e-01,
         1.90458756e-01, -9.07114668e-02,  9.04362318e-02,  8.31346798e-02,
         2.13160961e-02,  2.15432298e-02, -3.73879676e-03,  1.02859274e-01,
        -1.92061684e-02,  4.74695502e-02, -1.62588613e-01, -1.35483660e-01,
         1.82927643e-01, -2.60349267e-01,  7.40103543e-05,  4.42716077e-02,
         2.61492994e-01,  1.87196986e-01,  1.68907708e-01,  2.87977955e-01,
         3.45983751e-01, -9.27126533e-02, -5.13456844e-02,  2.33013103e-01,
         1.00735605e-01, -9.60286769e-02, -3.34088736e-02,  1.02897562e-03,
         2.79550907e-01,  1.86108535e-01,  1.78718855e-01, -3.16633931e-01,
         1.750